# 01 - Data Validation

Audit all three raw tables before data cleaning and analysis

The goal is to identify and document data quality issues before data cleaning

In [1]:
# Setup
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from settings import PATHS, apply_pandas_settings

apply_pandas_settings()

In [2]:
# Load data
customers = pd.read_csv(PATHS['raw_customers'])
loans = pd.read_csv(PATHS['raw_loans'])
transactions = pd.read_csv(PATHS['raw_transactions'])

In [3]:
customers.head()

,customer_id,account_open_date,acquisition_channel,kyc_level,location
0,CUS00001,2022-01-05,Ads,Tier 2,Lagos
1,CUS00002,2022-01-10,Loan Campaign,Tier 2,Lagos
2,CUS00003,2022-01-22,Loan Campaign,Tier 2,Lagos
3,CUS00004,2022-05-20,Ads,Tier 3,Lagos
4,CUS00005,2022-02-27,Loan Campaign,Tier 1,Plateau


In [4]:
loans.head()

,loan_id,customer_id,loan_taken,loan_amount,loan_defaulted
0,LON00001,CUS00001,No,NaN,NaN
1,LON00002,CUS00002,Yes,40000.00,No
2,LON00003,CUS00003,Yes,10000.00,No
3,LON00004,CUS00004,No,NaN,NaN
4,LON00005,CUS00005,Yes,20000.00,Yes


In [5]:
transactions.head()

,transaction_id,customer_id,transaction_type,transaction_date,remark,amount
0,TXN0021277,CUS01170,Transfer,2022-05-14 07:18:26,Transfer to Blessing Okonkwo | Access Bank | 2...,111300.00
1,TXN0030504,CUS01668,Bank Charge,2022-08-27 13:19:08,ATM usage fee,35.00
2,TXN0052726,CUS02941,VAT,2022-07-12 11:13:38,VAT on transaction fee,7.50
3,TXN0044119,CUS02475,Airtime,2022-06-29 07:39:20,9mobile airtime recharge to 08124622954,750.00
4,TXN0017473,CUS00944,VAT,2022-07-01 09:26:20,VAT on transaction fee,7.50


## Schema Check

In [6]:
# Schema check (shape, column names, and data types)
print('CUSTOMERS')
display(customers.dtypes)
print(f'Shape: {customers.shape}')

print('\nLOANS')
display(loans.dtypes)
print(f'Shape: {loans.shape}')

print('\nTRANSACTIONS')
display(transactions.dtypes)
print(f'Shape: {transactions.shape}')

CUSTOMERS


customer_id            str
account_open_date      str
acquisition_channel    str
kyc_level              str
location               str
dtype: object

Shape: (5000, 5)

LOANS


loan_id               str
customer_id           str
loan_taken            str
loan_amount       float64
loan_defaulted        str
dtype: object

Shape: (5000, 5)

TRANSACTIONS


transaction_id          str
customer_id             str
transaction_type        str
transaction_date        str
remark                  str
amount              float64
dtype: object

Shape: (88869, 6)


### Schema Check - Findings

| Table | Column | Current Type | Expected Type | Action Required |
|---|---|---|---|---|
| customers | account_open_date | object (str) | datetime | Convert in cleaning |
| transactions | transaction_date | object (str) | datetime | Convert in cleaning |

All other columns loaded with correct types.

## Null Value Check

In [7]:
# Null value check
for name, df in [('CUSTOMERS', customers), ('LOANS', loans), ('TRANSACTIONS', transactions)]:
    print(f'{name}')
    nulls = df.isnull().sum()
    summary = pd.DataFrame({
        'null_count': nulls
    })

    display(summary[summary['null_count'] > 0])

CUSTOMERS


,null_count


LOANS


,null_count
loan_amount,2205
loan_defaulted,2175


TRANSACTIONS


,null_count


### Null Check — Findings

| Table | Column | Null Count | Observation |
|---|---|---|---|
| loans | loan_amount | 2,205 | Null values present — consistent with customers who did not take a loan |
| loans | loan_defaulted | 2,175 | Null count does not match loan_amount — 30-row discrepancy flagged for investigation |

## Duplicate Check

In [8]:
# Duplicate check
for name, df, col_id in [
    ('CUSTOMERS', customers, 'customer_id'),
    ('LOANS', loans, 'loan_id'),
    ('TRANSACTIONS', transactions, 'transaction_id'),
]:
    dup_ids = df[col_id].duplicated().sum()
    dup_rows = df.duplicated().sum()

    print(f'{name}')
    print(f'Duplicate {col_id}s : {dup_ids:,}')
    print(f'Duplicate rows : {dup_rows:,}\n')

CUSTOMERS
Duplicate customer_ids : 0
Duplicate rows : 0

LOANS
Duplicate loan_ids : 0
Duplicate rows : 0

TRANSACTIONS
Duplicate transaction_ids : 31
Duplicate rows : 31



In [9]:
# Sample duplicate transactions
dup_mask = transactions.duplicated(keep=False)
dup_sample = transactions[dup_mask].sort_values('transaction_id').head(7)
display(dup_sample)

,transaction_id,customer_id,transaction_type,transaction_date,remark,amount
3062,TXN0009018,CUS00495,Bill Pay,2022-11-27 13:06:35,DSTV Subscription payment,17050.00
8996,TXN0009018,CUS00495,Bill Pay,2022-11-27 13:06:35,DSTV Subscription payment,17050.00
71775,TXN0009018,CUS00495,Bill Pay,2022-11-27 13:06:35,DSTV Subscription payment,17050.00
388,TXN0010280,CUS00561,Transfer,2022-11-30 12:38:09,Transfer to Blessing Ihejirika | Sterling Bank...,83200.00
58001,TXN0010280,CUS00561,Transfer,2022-11-30 12:38:09,Transfer to Blessing Ihejirika | Sterling Bank...,83200.00
59182,TXN0010627,CUS00578,Savings,2022-07-29 00:10:54,Target savings contribution,5900.00
33608,TXN0010627,CUS00578,Savings,2022-07-29 00:10:54,Target savings contribution,5900.00


### Duplicate Check — Findings

| Table | Duplicate IDs | Duplicate Rows | Observation |
|---|---|---|---|
| customers | 0 | 0 | Clean — no duplicates |
| loans | 0 | 0 | Clean — no duplicates |
| transactions | 31 | 31 | 31 fully duplicated rows found across multiple transaction IDs |

**Issue identified:** 31 duplicate rows in the transactions table.
Each duplicate is an exact copy of an existing row — same transaction_id, customer_id,
transaction_type, transaction_date, amount, and remark.
Duplicates are scattered randomly throughout the dataset.
If left uncleaned, these rows will inflate transaction counts and amounts for affected customers,
potentially pushing them into the wrong segment.

**Action:** Remove duplicate rows during cleaning, keeping the first occurrence of each.

## Value Consistency Check

In [10]:
# Value consistency check
tables_cols_dict = {
    'CUSTOMERS': ['acquisition_channel', 'kyc_level', 'location'],
    'LOANS': ['loan_taken', 'loan_defaulted'],
    'TRANSACTIONS': ['transaction_type'],
}

dfs = {
    'CUSTOMERS': customers,
    'LOANS': loans,
    'TRANSACTIONS': transactions,
}

for table, cols in tables_cols_dict.items():
    print(f'{table}')
    for col in cols:
        print(f'\n {col}:')
        display(dfs[table][col].value_counts(dropna=False))
    print()


CUSTOMERS

 acquisition_channel:


acquisition_channel
Loan Campaign     1847
Ads               1350
Referral           686
Agent Network      402
Direct             315
loan campaign       49
LOAN CAMPAIGN       42
Ad                  40
Loan Campaign       39
 Loan Campaign      37
ADS                 31
ads                 26
Ads                 21
REFERRAL            18
referral            16
Ref                 13
Agnt Ntwk           12
AGENT NETWORK        9
Agent Network        9
DIRECT               9
Direct               8
Referral             8
agent network        7
Dir                  5
direct               1
Name: count, dtype: int64


 kyc_level:


kyc_level
Tier 1    2557
Tier 2    1926
Tier 3     517
Name: count, dtype: int64


 location:


...

loan_taken
Yes    2795
No     2205
Name: count, dtype: int64


 loan_defaulted:


loan_defaulted
NaN    2175
Yes    1428
No     1397
Name: count, dtype: int64


TRANSACTIONS

 transaction_type:


transaction_type
VAT                   21935
Savings               13034
Airtime               13008
Bill Pay              12995
Transfer               7683
Transfer Fee           7672
Stamp Duty             7212
Reversal               1501
Card Payment - POS     1280
Card Payment - ATM     1276
Bank Charge            1273
Name: count, dtype: int64

### Value Consistency Check — Findings

| Table | Column | Status | Observation |
|---|---|---|---|
| customers | acquisition_channel | Issue | 5 valid values appearing in 25 different forms — case variations (loan campaign, LOAN CAMPAIGN), abbreviations (Ref, Agnt Ntwk, Dir), leading and trailing whitespace |
| customers | kyc_level | Clean | 3 valid values, consistently formatted |
| customers | location | Clean | 3 valid values, consistently formatted |
| loans | loan_taken | Clean | Yes/No only, consistently formatted |
| loans | loan_defaulted | Issue | 2,825 non-null values but only 2,795 customers took a loan — 30 rows have loan_defaulted populated where loan_taken = No |
| transactions | transaction_type | Clean | 11 valid transaction types, consistently formatted |

**Issues to fix in data cleaning:**
1. Standardise acquisition_channel — strip whitespace, fix casing, expand abbreviations
2. Investigate and resolve the 30-row loan_defaulted contradiction

## Cross-Table Check — Referential Integrity

In [11]:
# Referential integrity check
valid_customer_ids = set(customers['customer_id'])

loans_orphans = loans[~loans['customer_id'].isin(valid_customer_ids)]

transactions_orphans = transactions[~transactions['customer_id'].isin(valid_customer_ids)]

print('customer ids in loans table, but not in customers table')
print(f' Count: {len(loans_orphans):,}')

print('\n customer ids in transactions table, but not in customers table')
print(f' Count: {len(transactions_orphans):,}')

no_loan_customers = customers[~customers['customer_id'].isin(loans['customer_id'])]
print('\n customers with no loan records')
print(f' Count: {len(no_loan_customers):,}')

customer ids in loans table, but not in customers table
 Count: 0

 customer ids in transactions table, but not in customers table
 Count: 0

 customers with no loan records
 Count: 0


### Referential Integrity Check — Findings

| Check | Count | Status |
|---|---|---|
| Loans with customer_id not in customers table | 0 | Clean |
| Transactions with customer_id not in customers table | 0 | Clean |
| Customers with no loan record | 0 | Clean |

All three tables are fully consistent with each other.
Every customer_id in loans and transactions exists in the customers table.
Every customer has exactly one loan record.

## Outlier Check — Transaction Amounts

In [12]:
# Outlier check
transactions['transaction_type'].value_counts()

transaction_type
VAT                   21935
Savings               13034
Airtime               13008
Bill Pay              12995
Transfer               7683
Transfer Fee           7672
Stamp Duty             7212
Reversal               1501
Card Payment - POS     1280
Card Payment - ATM     1276
Bank Charge            1273
Name: count, dtype: int64

In [13]:
system_transaction_types = [
    'VAT', 'Transfer Fee', 'Stamp Duty', 'Reversal', 'Bank Charge'
]

user_transactions = transactions[~transactions['transaction_type'].isin(system_transaction_types)]
stats = transactions.groupby('transaction_type')['amount'].describe().round(2)

display(stats)

,count,mean,std,min,25%,50%,75%,max
transaction_type,,,,,,,,
Airtime,13008.00,1320.94,1551.53,100.00,650.00,1150.00,1700.00,120000.00
Bank Charge,1273.00,35.00,0.00,35.00,35.00,35.00,35.00,35.00
Bill Pay,12995.00,25692.21,19388.00,1000.00,13275.00,25400.00,37800.00,1200000.00
Card Payment - ATM,1276.00,75339.66,43096.83,2050.00,37037.50,73250.00,113750.00,149900.00
Card Payment - POS,1280.00,101995.04,56522.50,1050.00,52800.00,104400.00,149450.00,199700.00
Reversal,1501.00,52040.17,42541.64,900.00,18900.00,39350.00,79500.00,199700.00
Savings,13034.00,51293.06,73137.76,500.00,25150.00,49675.00,74850.00,4500000.00
Stamp Duty,7212.00,50.00,0.00,50.00,50.00,50.00,50.00,50.00
Transfer,7683.00,75294.29,62518.64,1000.00,36800.00,73600.00,111775.00,3000000.00


In [14]:
# Identify outliers using IQR method
outlier_rows = []

for txn_type in user_transactions['transaction_type'].unique():
    txn_type_amount = user_transactions[user_transactions['transaction_type'] == txn_type]['amount']
    Q1 = txn_type_amount.quantile(0.25)
    Q3 = txn_type_amount.quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 3 * IQR
    outliers = user_transactions[
        (user_transactions['transaction_type'] == txn_type) & (user_transactions['amount'] > upper_bound)
    ]
    if len(outliers) > 0:
        print(f'{txn_type}: upper_bound = #{upper_bound:,.2f} | outliers found = {len(outliers)}')
        outlier_rows.append(outliers)

if outlier_rows:
    all_outliers = pd.concat(outlier_rows)
else:
    all_outliers = pd.DataFrame()
    
print(f'\nTotal outlier rows : {len(all_outliers)}')
display(all_outliers[['transaction_id', 'customer_id', 'transaction_type', 'amount']].sort_values('transaction_type'))



Transfer: upper_bound = #336,700.00 | outliers found = 4
Airtime: upper_bound = #4,850.00 | outliers found = 60
Bill Pay: upper_bound = #111,375.00 | outliers found = 3
Savings: upper_bound = #223,950.00 | outliers found = 6

Total outlier rows : 73


,transaction_id,customer_id,transaction_type,amount
44441,TXN0054255,CUS03032,Airtime,4950.00
42181,TXN0010939,CUS00592,Airtime,4900.00
47395,TXN0004325,CUS00244,Airtime,4900.00
48898,TXN0081678,CUS04575,Airtime,4950.00
49376,TXN0065663,CUS03673,Airtime,5000.00
56611,TXN0032179,CUS01757,Airtime,4950.00
57795,TXN0038218,CUS02127,Airtime,5000.00
57826,TXN0009962,CUS00548,Airtime,5000.00
61097,TXN0080450,CUS04504,Airtime,4950.00
61361,TXN0079237,CUS04443,Airtime,4950.00


### Outlier Check — Findings

**Method:** IQR method (multiplier = 3) applied per transaction type

**IQR results:**
| Transaction Type | Upper Bound | Rows Flagged |
|---|---|---|
| Airtime | ₦4,850 | 60 |
| Bill Pay | ₦111,375 | 3 |
| Savings | ₦223,950 | 6 |
| Transfer | ₦336,700 | 4 |

**Limitation identified:** The IQR method over-flagged airtime transactions.
58 of the 60 flagged airtime rows fall between ₦4,900–₦5,000 — amounts that are
statistically extreme given the skewed airtime distribution but behaviourally
legitimate. People purchase airtime for multiple lines and in bulk.

**Manual inspection conclusion:**
Genuinely erroneous transactions are identifiable by the magnitude of deviation
from typical values — not just statistical position. The following rows are
considered true outliers based on the degree of deviation and business context:

| Transaction ID | Customer | Type | Amount | Why Flagged |
|---|---|---|---|---|
| TXN9000013 | CUS04543 | Airtime | ₦75,000 | 15x the typical maximum |
| TXN9000014 | CUS01999 | Airtime | ₦120,000 | 24x the typical maximum |
| TXN9000010 | CUS00990 | Bill Pay | ₦500,000 | 10x the typical maximum |
| TXN9000011 | CUS04260 | Bill Pay | ₦850,000 | 17x the typical maximum |
| TXN9000012 | CUS00641 | Bill Pay | ₦1,200,000 | 24x the typical maximum |
| TXN9000000 | CUS00609 | Savings | ₦800,000 | 8x the typical maximum |
| TXN9000001 | CUS01400 | Savings | ₦1,200,000 | 12x the typical maximum |
| TXN9000002 | CUS00970 | Savings | ₦2,500,000 | 25x the typical maximum |
| TXN9000003 | CUS02059 | Savings | ₦3,800,000 | 38x the typical maximum |
| TXN9000004 | CUS01497 | Savings | ₦4,200,000 | 42x the typical maximum |
| TXN9000005 | CUS03021 | Savings | ₦4,500,000 | 45x the typical maximum |
| TXN9000006 | CUS03653 | Transfer | ₦900,000 | 6x the typical maximum |
| TXN9000007 | CUS01522 | Transfer | ₦1,500,000 | 10x the typical maximum |
| TXN9000008 | CUS04364 | Transfer | ₦2,200,000 | 15x the typical maximum |
| TXN9000009 | CUS00651 | Transfer | ₦3,000,000 | 20x the typical maximum |

**Total confirmed outliers:** 15 rows

**Action:** Flag and remove these 15 rows in the cleaning notebook.
The remaining 58 statistically flagged airtime rows are retained as legitimate transactions.

## Validation Summary

### Issues Found

| # | Table | Column | Issue | Severity | Action |
|---|---|---|---|---|---|
| 1 | customers | account_open_date | Loaded as string, should be datetime | Medium | Convert to datetime in cleaning |
| 2 | transactions | transaction_date | Loaded as string, should be datetime | High | Convert to datetime in cleaning |
| 3 | customers | acquisition_channel | 5 valid values appearing in 25 different forms — wrong case, abbreviations, whitespace | High | Standardise in cleaning |
| 4 | loans | loan_defaulted | 30 rows have loan_defaulted = Yes where loan_taken = No — logical impossibility | High | Investigate and correct in cleaning |
| 5 | transactions | transaction_id | 31 fully duplicated rows scattered throughout the dataset | High | Remove duplicates in cleaning |
| 6 | transactions | amount | 15 genuinely erroneous outlier amounts identified across Airtime, Bill Pay, Savings, and Transfer — magnitudes between 6x and 45x typical values | High | Remove outlier rows in cleaning |

### Checks Passed

| Check | Observation |
|---|---|
| Null values | Nulls found only in loan_amount and loan_defaulted — consistent with customers who did not take a loan |
| Duplicate IDs and rows | customers and loans tables are fully clean |
| Categorical consistency | kyc_level, location, loan_taken, and transaction_type all contain consistent, correctly formatted values |
| Referential integrity | All customer_ids in loans and transactions match the customers table — no orphaned records found |

### Summary

- **6 issues identified** across all three tables
- **4 issues are high severity** — they will directly corrupt segmentation results if not cleaned
- **1 issue is medium severity** — wrong data type will break date arithmetic downstream
- All issues to be addressed in `02_data_cleaning.ipynb`